# General phase-matching pipeline

Repeatable version of the ad-hoc `PhaseMatching_Table.ipynb` investigation, built around one
lesson learned the hard way there: **raw mode index is not a stable identifier across geometry or
wavelength** (a mode's index has been observed to swing by 7+ just from a 50nm width change, and a
single mislabeled mode -- TM13 mistaken for TM40 -- cost an entire session of debugging). So the
workflow here puts ALL human visual confirmation up front, at one anchor geometry, and everything
after that runs unattended (potentially for hours) using overlap-based tracking that walks between
neighboring grid points rather than jumping straight from the anchor to every target.

**Workflow:**
1. Set the anchor geometry and materials.
2. Visually survey modes at the anchor (`visual_mode_survey`) -- review the plots, then fill in
   `modes_of_interest` with the confirmed index for each named mode.
3. (Optional) Auto-scan for *other* modes' crossings at the anchor (`scan_for_crossings`) -- surfaces
   candidates like TM02/TM20 without needing to know their index ahead of time. Visually confirm any
   candidates you want to add before including them in step 4.
4. Define the target geometry grid to cover.
5. Long automated run: for each named mode, walk its mode index across the whole grid
   (`walk_mode_across_points`), then compute the phase-matching crossing wavelength and
   width/height gradients at every grid point (`phase_matching_row`).
6. Assemble results into a table and save to CSV.

In [9]:
import numpy as np
import itertools
import emode_helpers as eh

# --- Anchor geometry + materials (edit for your design) ---
h_anchor = 350.0  # [nm]
w_anchor = 400.0  # [nm]
dx, dy = 10.0, 10.0

eq_o = '(1+2.8032/(1-0.015287/x**2)+0.36335/(1-0.036095/x**2)-33508000/(1+367200000/x**2))**0.5'
eq_e = '(1+0.017061/(1-0.043855/x**2)+3.1976/(1-0.022642/x**2)-57269000/(1-74226000/x**2))**0.5'
anisotropic_equation = f"[{eq_o},{eq_e},{eq_o}]"

## Step 1: visual mode survey at the anchor

Solves once at the anchor and plots each mode in `modes_to_plot` for you to review. Adjust the
range/wavelength below to cover whatever modes you're interested in -- widen it if the mode you
want isn't in the initial range. This is the ONLY step that needs your eyes on a plot; everything
after this cell is unattended.

In [10]:
survey_wavelength = 230.0  # [nm] -- pick something in the general region of interest
modes_to_plot = range(18, 35)  # narrowed for smoke test -- widen to range(0, 40) for a real new geometry
modes_to_plot = [23,30]
fdm_survey = eh.visual_mode_survey(anisotropic_equation, h_anchor, w_anchor, survey_wavelength,
                                    num_modes=45, modes_to_plot=modes_to_plot,
                                    x_resolution=dx, y_resolution=dy)

=== Survey at h=350.0, w=400.0, wavelength=230.0 (all plots below are this SAME geometry, only mode index varies) ===
-- mode 23 --  n_eff=2.15051
-- mode 30 --  n_eff=2.03437


## Step 1b: record confirmed mode indices

After reviewing the plots above, fill in the confirmed index for each named mode of interest.
`wavelengths_shg` should be wide enough to contain that mode's crossing across the ENTIRE target
grid you plan to sweep in step 4 -- not just at the anchor. A mode with a large `dWL/dwidth` (like
TM40, ~0.24 nm/nm) can shift its crossing wavelength by tens of nm across a wide grid, so give it a
generous window; a less width-sensitive mode (like TM04, ~-0.012 nm/nm) can use a narrower one.

`num_modes`/`window` for the walk should have headroom above the anchor index, since index has
been observed to shift by 7+ over a single 50nm step -- err generous, the walk is cheap per step.

In [12]:
modes_of_interest = {
    'TM04': dict(anchor_mode_idx=30, num_modes=60, window=20,
                 wavelengths_shg=np.arange(200, 300, 2.0), tracking_wavelength=228.244),
    'TM40': dict(anchor_mode_idx=23, num_modes=45, window=15,
                 wavelengths_shg=np.arange(200, 300, 2.0), tracking_wavelength=241.672),
}

## Step 2 (optional): auto-scan for other modes' crossings at the anchor

Looks for a phase-matching crossing across EVERY raw mode index at the anchor point, in exactly 2
EMode sweep calls (not one per index). This surfaces *candidates* like TM02/TM20 -- it does NOT
identify them by name. Visually confirm any index you want to keep (via `visual_mode_survey` with
`modes_to_plot=[that index]`) before adding it to `modes_of_interest` above and re-running from
Step 1b.

In [ ]:
scan_wavelengths = np.arange(160, 340, 2.0)
already_named = {cfg['anchor_mode_idx'] for cfg in modes_of_interest.values()}

candidates = eh.scan_for_crossings(anisotropic_equation, h_anchor, w_anchor, scan_wavelengths,
                                    fund_mode_idx=0, num_modes_fund=2, num_modes_scan=45,
                                    exclude_indices=already_named | {0},
                                    x_resolution=dx, y_resolution=dy)

print(f"Found {len(candidates)} candidate mode(s) with a crossing in "
      f"{scan_wavelengths[0]:.0f}-{scan_wavelengths[-1]:.0f}nm (excluding already-named modes):")
for mode_idx, info in sorted(candidates.items()):
    print(f"  mode {mode_idx}: crossing={info['wavelength_shg']:.3f}nm, n_eff={info['n_eff']:.5f}")

## Step 3: define the target geometry grid

Edit `h_values`/`w_values` to whatever range you want covered. The walk in Step 4 handles an
arbitrary set of (h, w) points, not just a rectangular grid, if you'd rather build `target_points`
some other way.

In [ ]:
# Smoke-test grid -- just 2 extra points beyond the anchor, to validate the pipeline cheaply
# before committing to a big sweep. Widen back out once this runs clean end-to-end.
h_values = [335.0, 350.0, 365.0]
w_values = [350.0, 375.0, 400.0, 425.0, 450.0, 475.0, 500.0, 525.0, 550.0  ]

target_points = [(h, w) for h, w in itertools.product(h_values, w_values)
                  if (h, w) != (h_anchor, w_anchor)]
print(f"{len(target_points)} target points queued.")

17 target points queued.


## Step 4: long run -- walk mode index across the grid, for each named mode

This is the unattended, potentially-hours-long part. Resilient to individual point failures (a
failed point doesn't stop the rest); progress prints as it goes so you can check in without
watching continuously.

In [8]:
import json
import os

walk_checkpoint_path = 'walked_indices_checkpoint.json'
walked_indices = {}  # mode_name -> {point: {'mode_idx', 'overlap', ...} or None}


def _serialize_walked(walked):
    """JSON-safe copy of walked_indices (tuple points/tracked_from -> lists)."""
    out = {}
    for mode_name, points in walked.items():
        out[mode_name] = {}
        for (h, w), info in points.items():
            if info is None:
                out[mode_name][f"{h},{w}"] = None
            else:
                info = dict(info)
                if info.get('tracked_from') is not None:
                    info['tracked_from'] = list(info['tracked_from'])
                out[mode_name][f"{h},{w}"] = info
    return out


def _deserialize_walked(raw):
    """Inverse of _serialize_walked: string point keys -> (h, w) tuples, tracked_from lists -> tuples."""
    walked = {}
    for mode_name, points in raw.items():
        walked[mode_name] = {}
        for key, info in points.items():
            h_str, w_str = key.split(',')
            point = (float(h_str), float(w_str))
            if info is not None:
                info = dict(info)
                if info.get('tracked_from') is not None:
                    info['tracked_from'] = tuple(info['tracked_from'])
            walked[mode_name][point] = info
    return walked


if os.path.exists(walk_checkpoint_path):
    with open(walk_checkpoint_path) as f:
        walked_indices = _deserialize_walked(json.load(f))
    print(f"Resuming from {walk_checkpoint_path}: "
          f"{ {m: len(p) for m, p in walked_indices.items()} } points loaded per mode")

for mode_name, cfg in modes_of_interest.items():
    # only points already SUCCESSFULLY solved (and not saturated) are skipped -- a previously-failed
    # point (None) is retried since the failure may have been transient (e.g. an EMode file-lock
    # error), and a saturated point is retried too since a wider num_modes on this run might resolve it
    already_solved = {p: info for p, info in walked_indices.get(mode_name, {}).items()
                       if info is not None and not info['saturated']}
    remaining_points = [p for p in target_points if p not in already_solved]

    if not remaining_points:
        print(f"\n=== {mode_name}: all {len(target_points)} points already solved, skipping ===")
        continue
    print(f"\n=== Walking {mode_name} across {len(remaining_points)}/{len(target_points)} "
          f"remaining points ===")

    def _checkpoint(point, info, solved, mode_name=mode_name):
        walked_indices[mode_name] = solved
        with open(walk_checkpoint_path, 'w') as f:
            json.dump(_serialize_walked(walked_indices), f, indent=2)

    walked_indices[mode_name] = eh.walk_mode_across_points(
        anisotropic_equation, (h_anchor, w_anchor), cfg['anchor_mode_idx'], remaining_points,
        tracking_wavelength=cfg['tracking_wavelength'], num_modes=cfg['num_modes'],
        window=cfg['window'], x_resolution=dx, y_resolution=dy, on_point=_checkpoint,
        resume_solved=already_solved)

print(f"\nWalk checkpoint saved to {walk_checkpoint_path}")

Resuming from walked_indices_checkpoint.json: {'TM04': 27, 'TM40': 27} points loaded per mode

=== Walking TM04 across 1/26 remaining points ===
(350.0, 550.0) -> (335.0, 550.0): {'mode_idx': 44, 'overlap': 0.8908252116331822, 'tracked_from': (350.0, 550.0), 'saturated': True}

=== TM40: all 26 points already solved, skipping ===

Walk checkpoint saved to walked_indices_checkpoint.json


## Step 5: phase-matching crossing + width/height gradient at every point

Uses each point's tracked mode index (from Step 4) directly -- no further tracking uncertainty.
Points where the walk failed or landed on a saturated (num_modes-ceiling) index are skipped with a
printed note rather than silently dropped.

In [ ]:
import csv
import os

partial_csv_path = 'general_pipeline_results_partial.csv'
# EMode's em.plot(file_name=...) fails with a relative path that includes a subdirectory (tested:
# a bare relative filename or a fully absolute path both work, but 'subdir/name' does not) -- use
# an absolute path here to sidestep that.
field_plots_dir = os.path.abspath('field_plots')
os.makedirs(field_plots_dir, exist_ok=True)

_numeric_fields = ('h_core', 'w_core', 'wavelength_shg', 'n_eff', 'dn_dWL_fund', 'dn_dWL_target',
                    'dn_dwidth_fund', 'dn_dwidth_target', 'dn_dheight_fund', 'dn_dheight_target',
                    'dWL_dwidth', 'dWL_dheight')

all_rows = []
done_keys = set()  # (mode_name, h_core, w_core) already computed and saved -- these are skipped

if os.path.exists(partial_csv_path) and os.path.getsize(partial_csv_path) > 0:
    with open(partial_csv_path, newline='') as f:
        for row in csv.DictReader(f):
            row['target_mode_idx'] = int(row['target_mode_idx'])
            for key in _numeric_fields:
                row[key] = float(row[key])
            all_rows.append(row)
            done_keys.add((row['target_mode_name'], row['h_core'], row['w_core']))
    print(f"Resuming from {partial_csv_path}: {len(all_rows)} rows already computed, "
          f"those points will be skipped.")

done_rows_by_key = {(r['target_mode_name'], r['h_core'], r['w_core']): r for r in all_rows}

total_considered = 0
_partial_f = open(partial_csv_path, 'a', newline='')
_partial_writer = csv.DictWriter(_partial_f, fieldnames=list(all_rows[0].keys())) if all_rows else None


def _append_row(row):
    global _partial_writer
    if _partial_writer is None:
        _partial_writer = csv.DictWriter(_partial_f, fieldnames=list(row.keys()))
        _partial_writer.writeheader()
    _partial_writer.writerow(row)
    _partial_f.flush()


def _save_plots(mode_name, h, w, mode_idx, wavelength_shg):
    """Save target+fundamental field plots for a matched point, skipping any that already exist on
    disk -- so re-running doesn't regenerate plots for points a previous run already recorded."""
    tag = f"{mode_name}_h{h:.1f}_w{w:.1f}"
    target_path = os.path.join(field_plots_dir, f"{tag}_target")
    fund_path = os.path.join(field_plots_dir, f"{tag}_fund")
    if not os.path.exists(target_path + '.png'):
        eh.save_mode_plot(anisotropic_equation, h, w, wavelength_shg, mode_idx,
                           num_modes=mode_idx + 10, x_resolution=dx, y_resolution=dy,
                           file_name=target_path)
    if not os.path.exists(fund_path + '.png'):
        eh.save_mode_plot(anisotropic_equation, h, w, 2 * wavelength_shg, 0,
                           num_modes=6, x_resolution=dx, y_resolution=dy, file_name=fund_path)


for mode_name, cfg in modes_of_interest.items():
    # the anchor point itself, using the originally-confirmed index
    points_and_indices = [((h_anchor, w_anchor), cfg['anchor_mode_idx'])]
    points_and_indices += [(p, info['mode_idx']) for p, info in walked_indices[mode_name].items()
                           if info is not None and not info['saturated']]
    total_considered += len(points_and_indices)

    for (h, w), mode_idx in points_and_indices:
        key = (mode_name, h, w)
        if key in done_keys:
            # already computed in a previous run -- just backfill a missing plot, if any, without
            # re-running the expensive phase_matching_row computation
            try:
                existing_row = done_rows_by_key[key]
                _save_plots(mode_name, h, w, existing_row['target_mode_idx'], existing_row['wavelength_shg'])
            except Exception as e:
                print(f"  WARNING: field plot backfill failed for {key}: {repr(e)}")
            continue
        print(f"{mode_name} @ h={h}, w={w}, mode={mode_idx} ...")
        try:
            row = eh.phase_matching_row(
                anisotropic_equation, h, w, cfg['wavelengths_shg'], target_mode_idx=mode_idx,
                target_mode_name=mode_name, fund_mode_idx=0, num_modes_fund=6,
                num_modes_target=mode_idx + 10, x_resolution=dx, y_resolution=dy)
            if row is None:
                print(f"  no crossing found in {cfg['wavelengths_shg'][0]:.0f}-"
                      f"{cfg['wavelengths_shg'][-1]:.0f}nm -- skipped")
            else:
                all_rows.append(row)
                _append_row(row)
                try:
                    _save_plots(mode_name, h, w, mode_idx, row['wavelength_shg'])
                except Exception as e:
                    print(f"  WARNING: field plot save failed: {repr(e)}")
        except Exception as e:
            print(f"  FAILED: {repr(e)}")

_partial_f.close()

print(f"\n{len(all_rows)} of {total_considered} points computed successfully "
      f"({len(done_keys)} resumed from a previous run).")
print(f"Rows saved incrementally to {partial_csv_path} as they were computed.")
print(f"Field plots saved to {field_plots_dir}/")

## Step 6: results table + save

In [15]:
print(eh.format_table(all_rows))

import csv
csv_path = 'general_pipeline_results.csv'
if all_rows:
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"\nSaved {len(all_rows)} rows to {csv_path}")

Mode         | h            | w            | PM WL (nm)   | n_eff        | dWL/dw       | dWL/dh      
------------------------------------------------------------------------------------------------------
TM04         | 350.0        | 400.0        | 228.829      | 2.05001      | -0.011937    | 0.209263    
TM04         | 335.0        | 400.0        | 225.787      | 2.04971      | -0.010860    | 0.209366    
TM04         | 335.0        | 375.0        | 226.073      | 2.04253      | -0.011930    | 0.211120    
TM04         | 350.0        | 375.0        | 229.091      | 2.04272      | -0.016839    | 0.215954    
TM04         | 335.0        | 350.0        | 226.405      | 2.03400      | -0.005932    | 0.219518    
TM04         | 350.0        | 350.0        | 229.355      | 2.03411      | 0.014763     | 0.195787    
TM04         | 335.0        | 425.0        | 225.599      | 2.05571      | 0.105459     | 0.054008    
TM04         | 350.0        | 425.0        | 228.541      | 2.05630      

## Step 7: analytical surface fits

Two kinds of surface, both fit as low-order (quadratic) polynomials via least squares:

1. `n(WL, h, w)` for each of TM00 (fundamental), TM04, TM40 -- effective index as a function of
   wavelength, height, and width.
2. `WL_pm(h, w)` for TM04 and TM40 -- the phase-matching wavelength as a function of height and
   width (where the two `n(WL, h, w)` surfaces for TM00 and each target mode intersect).

Every row in the results table carries not just a value but the *local gradients* EMode already
computed (`dn_dwidth`, `dn_dheight`, `dn_dWL`, `dWL_dwidth`, `dWL_dheight`) -- the fits below use a
Hermite-style least squares that matches both value AND gradient at each sample point, which with
only ~26-50 points spread over a 2-3 variable domain gives a much better-conditioned fit than
throwing the gradients away and fitting values alone.

Also note: by construction, at the phase-matching wavelength the fundamental's index EQUALS the
target mode's index (that's what "crossing" means) -- so every TM04/TM40 row also gives a free
data point (with its own gradients) for the TM00 surface at wavelength `2 * wavelength_shg`.

In [ ]:
import csv
import itertools


def load_results_table(csv_path='general_pipeline_results.csv'):
    """Reloaded fresh from disk so this analysis can be re-run independently of the EMode pipeline."""
    numeric_fields = ('h_core', 'w_core', 'wavelength_shg', 'n_eff', 'dn_dWL_fund', 'dn_dWL_target',
                       'dn_dwidth_fund', 'dn_dwidth_target', 'dn_dheight_fund', 'dn_dheight_target',
                       'dWL_dwidth', 'dWL_dheight')
    rows = []
    with open(csv_path, newline='') as f:
        for row in csv.DictReader(f):
            row['target_mode_idx'] = int(row['target_mode_idx'])
            for key in numeric_fields:
                row[key] = float(row[key])
            rows.append(row)
    return rows


results_table = load_results_table()
print(f"Loaded {len(results_table)} rows from general_pipeline_results.csv")


def poly_terms(n_vars, degree):
    """All exponent-tuples (monomials) in n_vars variables with total degree <= `degree`."""
    terms = []
    for total in range(degree + 1):
        for exps in itertools.product(range(total + 1), repeat=n_vars):
            if sum(exps) == total:
                terms.append(exps)
    return terms


def _eval_monomial(exps, point):
    v = 1.0
    for e, x in zip(exps, point):
        v *= x ** e
    return v


def _eval_monomial_deriv(exps, point, var_idx):
    e = exps[var_idx]
    if e == 0:
        return 0.0
    new_exps = list(exps)
    new_exps[var_idx] -= 1
    v = float(e)
    for ee, x in zip(new_exps, point):
        v *= x ** ee
    return v


class PolyFit:
    """A low-order polynomial surface fit, solved in centered/scaled coordinates for conditioning
    but callable directly in original units: fit(wl, h, w) -> predicted value."""

    def __init__(self, var_names, terms, coeffs, centers, scales, rms_value_residual):
        self.var_names = var_names
        self.terms = terms
        self.coeffs = coeffs
        self.centers = centers
        self.scales = scales
        self.rms_value_residual = rms_value_residual

    def __call__(self, *point):
        p_norm = tuple((x - c) / s for x, c, s in zip(point, self.centers, self.scales))
        return sum(c * _eval_monomial(t, p_norm) for c, t in zip(self.coeffs, self.terms))

    def summary(self):
        lines = [f"fit RMS residual on values: {self.rms_value_residual:.3e}"]
        for t, c in zip(self.terms, self.coeffs):
            factors = '*'.join(f"{n}'^{e}" for n, e in zip(self.var_names, t) if e > 0) or '1'
            lines.append(f"  {c:+.6g} * {factors}")
        lines.append("  (primed variables are normalized: x' = (x - center) / scale)")
        for n, c, s in zip(self.var_names, self.centers, self.scales):
            lines.append(f"  {n}: center={c:.4g}, scale={s:.4g}")
        return '\n'.join(lines)


def fit_hermite_polynomial(samples, degree, var_names):
    """samples: list of {'point': (x1,...,xn), 'value': f, 'grad': (df/dx1,...,df/dxn) or None}.
    Least-squares fit against BOTH the value and the known local gradient at every sample point.
    """
    n_vars = len(samples[0]['point'])
    points = np.array([s['point'] for s in samples], dtype=float)
    centers = points.mean(axis=0)
    scales = points.std(axis=0)
    scales[scales == 0] = 1.0

    terms = poly_terms(n_vars, degree)
    rows, targets = [], []
    for s in samples:
        p_norm = tuple((x - c) / sc for x, c, sc in zip(s['point'], centers, scales))
        rows.append([_eval_monomial(t, p_norm) for t in terms])
        targets.append(s['value'])
        grad = s.get('grad')
        if grad is not None:
            for var_idx, g in enumerate(grad):
                if g is None:
                    continue
                # chain rule: d/dx'_i = scale_i * d/dx_i, since x_i = center_i + scale_i * x'_i
                rows.append([_eval_monomial_deriv(t, p_norm, var_idx) for t in terms])
                targets.append(g * scales[var_idx])

    A = np.array(rows)
    b = np.array(targets)
    coeffs, *_ = np.linalg.lstsq(A, b, rcond=None)

    value_rows = np.array([[_eval_monomial(t, tuple((x - c) / sc for x, c, sc in zip(s['point'], centers, scales)))
                             for t in terms] for s in samples])
    value_targets = np.array([s['value'] for s in samples])
    rms_value_residual = float(np.sqrt(np.mean((value_rows @ coeffs - value_targets) ** 2)))

    return PolyFit(var_names, terms, coeffs, centers, scales, rms_value_residual)


def index_samples(rows, mode):
    """(WL, h, w) -> n_eff samples with gradients for the named target mode, or 'TM00' for the
    fundamental (whose index at 2*wavelength_shg equals n_eff by the phase-matching condition
    itself -- so every TM04/TM40 row also gives a free fundamental-mode data point)."""
    samples = []
    for row in rows:
        if mode == 'TM00':
            wl = 2 * row['wavelength_shg']
            grad = (row['dn_dWL_fund'], row['dn_dheight_fund'], row['dn_dwidth_fund'])
        elif row['target_mode_name'] == mode:
            wl = row['wavelength_shg']
            grad = (row['dn_dWL_target'], row['dn_dheight_target'], row['dn_dwidth_target'])
        else:
            continue
        samples.append({'point': (wl, row['h_core'], row['w_core']), 'value': row['n_eff'], 'grad': grad})
    return samples


index_fits = {}
for mode in ('TM00', 'TM04', 'TM40'):
    samples = index_samples(results_table, mode)
    fit = fit_hermite_polynomial(samples, degree=2, var_names=('WL', 'h', 'w'))
    index_fits[mode] = fit
    print(f"\n=== n_{mode}(WL, h, w) -- fit from {len(samples)} points ===")
    print(fit.summary())

In [ ]:
def wl_samples(rows, mode):
    """(h, w) -> wavelength_shg samples with gradients for the named target mode."""
    return [{'point': (row['h_core'], row['w_core']), 'value': row['wavelength_shg'],
             'grad': (row['dWL_dheight'], row['dWL_dwidth'])}
            for row in rows if row['target_mode_name'] == mode]


wl_fits = {}
for mode in ('TM04', 'TM40'):
    samples = wl_samples(results_table, mode)
    fit = fit_hermite_polynomial(samples, degree=2, var_names=('h', 'w'))
    wl_fits[mode] = fit
    print(f"\n=== WL_pm_{mode}(h, w) -- fit from {len(samples)} points ===")
    print(fit.summary())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# validated CVD-safe categorical pair (worst-case Delta-E 24.7 protan / 33.6 normal-vision)
COLOR_TM04 = '#2a78d6'  # blue
COLOR_TM40 = '#eb6834'  # orange

h_all = [row['h_core'] for row in results_table]
w_all = [row['w_core'] for row in results_table]
h_grid = np.linspace(min(h_all), max(h_all), 40)
w_grid = np.linspace(min(w_all), max(w_all), 40)
H, W = np.meshgrid(h_grid, w_grid)

WL_TM04 = np.vectorize(lambda h, w: wl_fits['TM04'](h, w))(H, W)
WL_TM40 = np.vectorize(lambda h, w: wl_fits['TM40'](h, w))(H, W)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(H, W, WL_TM04, color=COLOR_TM04, alpha=0.7, edgecolor='none')
ax.plot_surface(H, W, WL_TM40, color=COLOR_TM40, alpha=0.7, edgecolor='none')

# overlay the actual simulated points for reference
tm04_pts = [(r['h_core'], r['w_core'], r['wavelength_shg']) for r in results_table if r['target_mode_name'] == 'TM04']
tm40_pts = [(r['h_core'], r['w_core'], r['wavelength_shg']) for r in results_table if r['target_mode_name'] == 'TM40']
if tm04_pts:
    ax.scatter(*zip(*tm04_pts), color=COLOR_TM04, s=15, depthshade=False)
if tm40_pts:
    ax.scatter(*zip(*tm40_pts), color=COLOR_TM40, s=15, depthshade=False)

ax.set_xlabel('h (nm)')
ax.set_ylabel('w (nm)')
ax.set_zlabel('phase-matching wavelength (nm)')
ax.set_title('Phase-matching wavelength surfaces: TM04 vs TM40')
ax.legend(handles=[Patch(color=COLOR_TM04, label='TM04'), Patch(color=COLOR_TM40, label='TM40')],
          loc='upper left')

fig.tight_layout()
fig_path = 'phase_matching_surfaces.png'
fig.savefig(fig_path, dpi=150)
print(f"Saved figure to {fig_path}")
plt.show()

In [ ]:
def _poly_add(p1, p2):
    out = dict(p1)
    for k, v in p2.items():
        out[k] = out.get(k, 0.0) + v
    return out


def _poly_scale(p, s):
    return {k: v * s for k, v in p.items()}


def _poly_mul(p1, p2):
    out = {}
    for k1, v1 in p1.items():
        for k2, v2 in p2.items():
            k = tuple(a + b for a, b in zip(k1, k2))
            out[k] = out.get(k, 0.0) + v1 * v2
    return out


def _affine_var_power(n_vars, var_idx, center, scale, power):
    """Polynomial (in raw units) for ((x_i - center) / scale)^power."""
    zero = tuple(0 for _ in range(n_vars))
    one_hot = tuple(1 if j == var_idx else 0 for j in range(n_vars))
    base = {one_hot: 1.0 / scale, zero: -center / scale}
    result = {zero: 1.0}
    for _ in range(power):
        result = _poly_mul(result, base)
    return result


def raw_unit_terms(fit):
    """Expand a PolyFit's normalized-coordinate fit into a {exponent_tuple: coeff} polynomial in
    the ORIGINAL units (e.g. plain w, h, WL in nm) -- exact, not a re-fit, so it carries no new
    numerical error (verified to agree with PolyFit.__call__ to ~1e-14)."""
    n_vars = len(fit.var_names)
    zero = tuple(0 for _ in range(n_vars))
    raw = {}
    for term_exps, coeff in zip(fit.terms, fit.coeffs):
        monomial = {zero: 1.0}
        for var_idx, e in enumerate(term_exps):
            if e == 0:
                continue
            monomial = _poly_mul(monomial, _affine_var_power(n_vars, var_idx, fit.centers[var_idx],
                                                               fit.scales[var_idx], e))
        raw = _poly_add(raw, _poly_scale(monomial, coeff))
    return raw


def format_formula(fit, lhs):
    """Human-readable 'lhs(vars) = ...' string in raw (original) units, highest-degree terms first."""
    raw = raw_unit_terms(fit)
    ordered = sorted(raw.items(), key=lambda kv: (-sum(kv[0]), kv[0]))
    parts = []
    for exps, coeff in ordered:
        factors = ''.join(f"*{n}" if e == 1 else f"*{n}^{e}" for n, e in zip(fit.var_names, exps) if e > 0)
        parts.append(f"{coeff:+.6g}{factors}")
    return f"{lhs} = " + " ".join(parts)


print("=== Index surfaces (nm in, dimensionless index out) ===")
for mode, fit in index_fits.items():
    print(f"[RMS residual {fit.rms_value_residual:.2e}]")
    print(format_formula(fit, f"n_{mode}(WL, h, w)"))
    print()

print("=== Phase-matching wavelength surfaces (nm in, nm out) ===")
for mode, fit in wl_fits.items():
    print(f"[RMS residual {fit.rms_value_residual:.2e}]")
    print(format_formula(fit, f"WL_pm_{mode}(h, w)"))
    print()